# Two exact Fourier papers: a numerical playground

Edit the next cell, then **Run All**. Defaults stay small and run on CPU; choose `DEVICE="gpu"` in a fresh kernel for CUDA. This notebook ships without saved outputs.

- [Finite tensor savings and exact Fourier circuits](https://github.com/openai/math/blob/adc7f1241b42e322a6451854ab7e4b4c146bf78a/preprints/Finite-tensor-savings-and-exact-Fourier-circuits-September-25-2026/main.pdf): the original Lean theorem gives exact scalar-circuit savings along an unbounded sequence of lengths. The repository also verifies the companion's explicit finite saving seed in Lean. Its full width is infeasible to materialize.
- [An explicit power saving for the exact discrete Fourier transform](https://github.com/openai/math/blob/adc7f1241b42e322a6451854ab7e4b4c146bf78a/preprints/An-explicit-power-saving-for-the-exact-discrete-Fourier-transform-September-25-2026/main.pdf): explore the local Newton factorization and proposed all-length cost exponent. **The stronger uniform algorithm theorem remains unfinished in this repository.** The dense Newton experiment is a numerical factor-identity test, not its fast implementation.

JAX is optional for the package, but required to run these numerical cells. Use an environment containing this repository, JAX/jaxlib and NumPy; matplotlib and ipywidgets are optional. No cell installs software or connects to another machine. Python 3.10+ is supported.


In [ ]:
from fractions import Fraction

# If opened outside this checkout, set its absolute path here (or set EXACT_FOURIER_REPO).
REPO_HINT = None
DEVICE = "cpu"               # "cpu", "gpu", or "auto"; change in a fresh kernel
COMPLEX_DTYPE = "complex128"   # also try "complex64"; x64 is enabled by the setup cell
SEED = 130

H = 100                      # symbolic combinatorial budget only; never allocate its full circuit
COLUMNS = None               # None chooses floor(2*H_calls/Delta)+1 when Delta > 0
SHEAR_COEFFICIENT = Fraction(1, 1)
SHEAR_TARGET_LOG2 = 20        # increase to observe loss of a small source during compiled shears
C_BITS = 6                   # ordinary C tensor on 2**C_BITS entries; bounded below
DFT_LENGTH = 32              # radix-two when possible, Bluestein for any positive length
NEWTON_LENGTH = 8            # dense companion identity, default guard <=64; increasing n can be unstable
WARM_REPEATS = 3
ENABLE_WIDGETS = False       # optional amplitude slider, only if ipywidgets is already installed

RUN_FULL_PROJECTED = False   # optional full 72,842-record toy quotient; can be slow on GPU
PROJECTED_BITS = 1           # try 1 or 4; always uses toy h=4, NOT the saving h=100 seed


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import sys
import time

hint = REPO_HINT or os.environ.get("EXACT_FOURIER_REPO")
candidates = ([Path(hint).expanduser().resolve()] if hint else []) + [Path.cwd(), *Path.cwd().parents]
REPO = next((p for p in candidates if (p / "src/exact_fourier/jax_backend.py").is_file()), None)
if REPO is None:
    raise RuntimeError("Set REPO_HINT to the checkout containing src/exact_fourier/jax_backend.py")
sys.path.insert(0, str(REPO / "src"))
if DEVICE not in ("cpu", "gpu", "auto"):
    raise ValueError("DEVICE must be cpu, gpu, or auto")
if DEVICE != "auto":
    os.environ["JAX_PLATFORMS"] = "cuda" if DEVICE == "gpu" else DEVICE
try:
    import jax
    import jax.numpy as jnp
    import numpy as np
except ImportError as exc:
    raise RuntimeError("Select a notebook kernel with JAX/jaxlib and NumPy installed") from exc
jax.config.update("jax_enable_x64", True)
chosen_device = jax.devices(None if DEVICE == "auto" else DEVICE)[0]
jax.config.update("jax_default_device", chosen_device)
if COMPLEX_DTYPE not in ("complex64", "complex128"):
    raise ValueError("Choose complex64 or complex128")
DTYPE = jnp.dtype(COMPLEX_DTYPE)

from exact_fourier.jax_backend import compile_c_tensor, compile_projected, compile_projected_target, compile_shear
from exact_fourier.jax_fft import compile_radix2, compile_bluestein
from exact_fourier.network import NetworkParameters
print({"repo": str(REPO), "jax": jax.__version__, "device": str(chosen_device),
       "dtype": COMPLEX_DTYPE, "x64_enabled": jax.config.jax_enable_x64})


## 1. Exact seed budget, with huge powers kept symbolic

The budget counts **forward C calls**, with monomials free; it is not a floating-point operation count or runtime forecast. Integer arithmetic below recomputes the paper's combinatorial formulas. The closed Lean saving word is the `h=100, f=6544863` instance; other parameter rows are arithmetic explorations, not newly verified Lean witnesses. The upstream theorem concerns scalar circuits, including charged diagonal scalings, via its exact transfer argument.

Write $m=h^3$, $W_*=2^r$, $S=W_*m-\Delta$, and $H_{\rm calls}$ for the pointwise C count. After factoring out $2^{mf-1}$, the construction and ordinary tensor have coefficients
$Sf+rW_*+2H_{\rm calls}$ and $W_*(mf+r)$. The saved coefficient is exactly $f\Delta-2H_{\rm calls}$.

Never evaluate `2**b`: the verified seed has $b=6544863000071$ address bits. Its full word and array are infeasible. The small projected experiment later is a different, executable quotient.


In [ ]:
from decimal import Decimal, localcontext

def seed_budget(h, columns=None):
    if type(h) is not int or not 4 <= h <= 1000:
        raise ValueError("Use an integer 4 <= h <= 1000 for symbolic exploration")
    p = NetworkParameters(h)
    r = (p.wires - 1).bit_length()
    padded = 1 << r             # only the modest role exponent, not the full address exponent
    minimal = (2 * p.pointwise_calls // p.delta + 1) if p.delta > 0 else None
    f = (minimal or 1) if columns is None else columns
    if type(f) is not int or f < 1:
        raise ValueError("columns must be a positive integer or None")
    b = p.m * f + r
    residual = padded * p.m - p.delta
    actual = residual * f + r * padded + 2 * p.pointwise_calls
    ordinary = padded * b
    saved = p.delta * f - 2 * p.pointwise_calls
    assert ordinary - actual == saved
    with localcontext() as context:
        context.prec = 80
        saved_fraction = Decimal(saved) / Decimal(ordinary)
        if p.delta > 0:
            epsilon = Decimal(p.delta) / Decimal(padded * p.m)
            # Stable 1-theta: subtracting a float approximation of theta from 1 loses digits.
            one_minus_theta = -(Decimal(1) - epsilon).ln() / Decimal(p.m).ln()
            theta = Decimal(1) - one_minus_theta
        else:
            epsilon = one_minus_theta = theta = None
        return {"h": h, "v": p.v, "degree": p.degree, "m": p.m,
                "W": p.wires, "W_star": padded, "r": r, "Delta": p.delta,
                "H_calls": p.pointwise_calls, "f": f, "minimal_saving_f": minimal,
                "b": b, "width": f"2^{b}", "common_factor": f"2^{p.m*f-1}",
                "actual_coefficient": actual, "ordinary_coefficient": ordinary,
                "saved_coefficient": saved, "saved_fraction": saved_fraction,
                "epsilon": epsilon, "one_minus_theta": one_minus_theta, "theta": theta}

budget = seed_budget(H, COLUMNS)
print(json.dumps(budget, indent=2, default=str))
print("Strict call saving:", budget["saved_coefficient"] > 0)
if H == 100 and COLUMNS is None:
    assert budget["f"] == 6544863 and budget["b"] == 6544863000071
    assert budget["saved_coefficient"] == 847159236000000
for h in (4, 20, 24, 100):
    row = seed_budget(h)
    print({k: str(row[k]) for k in ("h", "f", "b", "saved_fraction", "one_minus_theta")})


For the companion exponent, $\theta=\log_m(S/W_*)$ and
$1-\theta=-\ln(1-\Delta/(W_*m))/\ln m$.
The proposed all-length shape is $n(\ln n)^\theta(\ln\ln n)^{4-\theta}$.
The theorem's hidden constants and threshold are unknown here, so this shape cannot predict a crossover against an FFT. It is not a verified executable all-length scheduler in this repository.


In [ ]:
if budget["theta"] is not None:
    with localcontext() as context:
        context.prec = 80
        theta = budget["theta"]
        for n in (10**3, 10**6, 10**12):
            logn = Decimal(n).ln()
            loglogn = logn.ln()
            # Shape only, setting an UNKNOWN multiplicative constant to 1 for illustration.
            shape = Decimal(n) * (theta * logn.ln() + (4 - theta) * loglogn.ln()).exp()
            print({"n": n, "unit_constant_shape": str(shape),
                   "shape_over_n_log_n": str(shape / (Decimal(n) * logn))})
else:
    print("Delta <= 0: this toy parameter has no saving exponent from the formula.")


## 2. Literal three-C shear and loss of a small source

Both modes compute $(y,x)\mapsto(y+t x,x)$ exactly over $\mathbb Q(i)$.
The compiled mode runs the existing three-C word and its scalings; the direct mode computes the shear directly. XLA may fuse operations, so rounding depends on compiler/device/precision.
A large target can cause the compiled word to lose a small source while a whole-array relative error remains tiny. The **source absolute error** exposes that effect. This does not contradict the Lean theorem over mathematical complex numbers.


In [ ]:
shear_functions = {mode: compile_shear(SHEAR_COEFFICIENT, compiled_shears=mode) for mode in (True, False)}
coefficient = complex(float(SHEAR_COEFFICIENT.real), float(SHEAR_COEFFICIENT.imag))

def shear_measure(exponent, compiled):
    amplitude = 2.0 ** int(exponent)
    values = jnp.asarray([[amplitude + 0j], [1 + 0j]], dtype=DTYPE)
    result = np.asarray(shear_functions[compiled](values).block_until_ready())
    reference = np.asarray([[amplitude + coefficient], [1 + 0j]], dtype=np.complex128)
    return {"log2_target": int(exponent), "mode": "three_C" if compiled else "direct",
            "source": complex(result[1, 0]), "source_absolute_error": float(abs(result[1, 0] - 1)),
            "relative_l2": float(np.linalg.norm(result - reference) / np.linalg.norm(reference))}

def show_shear(exponent):
    for compiled in (True, False):
        print(shear_measure(exponent, compiled))

show_shear(SHEAR_TARGET_LOG2)
shear_curve = [shear_measure(exponent, compiled) for exponent in range(0, 81, 4) for compiled in (True, False)]
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
if plt is not None:
    fig, axis = plt.subplots(figsize=(7, 3))
    for mode in ("three_C", "direct"):
        rows = [row for row in shear_curve if row["mode"] == mode]
        axis.plot([r["log2_target"] for r in rows], [r["source_absolute_error"] for r in rows], label=mode)
    axis.set_yscale("symlog", linthresh=1e-12)
    axis.set(xlabel="log2(target amplitude), source=1", ylabel="source absolute error",
             title=f"Shear source restoration: {COMPLEX_DTYPE} on {chosen_device.platform}")
    axis.legend(); plt.show()


In [ ]:
if ENABLE_WIDGETS:
    try:
        import ipywidgets as widgets
    except ImportError:
        print("ipywidgets is absent; edit SHEAR_TARGET_LOG2 above and rerun the shear cell.")
    else:
        widgets.interact(show_shear, exponent=widgets.IntSlider(value=SHEAR_TARGET_LOG2,
                         min=0, max=80, step=1, description="log2 target", continuous_update=False))


## 3. Ordinary C tensor and optional complete projected network

$C=\begin{bmatrix}(1+i)/2&(1-i)/2\\(1-i)/2&(1+i)/2\end{bmatrix}$.
An ordinary $C^{\otimes k}$ is a Walsh-phase operator, **not the ordinary DFT**.
A random binary projection has its own phase $i^{\sum_j\langle L e_j,\xi\rangle}$; consequently the projected network must be compared with `compile_projected_target`, not generally with `compile_c_tensor(address_bits)`.

The optional full record loop below includes all dirty auxiliary/padding rows. Its `h=4` quotient has no claimed saving. Exact Gaussian-rational CPU tests and independently recorded CUDA runs are separate evidence from these interactive floating-point measurements.


In [ ]:
from exact_fourier.projection import BinaryProjection, ProjectedProgram, build_projected_program, projected_target_exact
from exact_fourier.scalars import GaussianRational as QI

def errors(actual, expected):
    actual, expected = np.asarray(actual), np.asarray(expected)
    residual = actual - expected
    scale = np.linalg.norm(expected)
    return {"finite": bool(np.all(np.isfinite(actual))),
            "relative_l2": float(np.linalg.norm(residual) / (scale if scale else 1)),
            "maximum_absolute": float(np.max(np.abs(residual), initial=0))}

def timed(function, values):
    started = time.perf_counter()
    result = function(values).block_until_ready()
    first = time.perf_counter() - started
    samples = []
    for _ in range(WARM_REPEATS):
        started = time.perf_counter()
        function(values).block_until_ready()
        samples.append(time.perf_counter() - started)
    return result, {"first_call_seconds": first, "warm_median_seconds": float(np.median(samples))}

if type(C_BITS) is not int or not 0 <= C_BITS <= 12:
    raise ValueError("Keep this C tensor experiment at 0 <= C_BITS <= 12")
projection = BinaryProjection(tuple(1 << bit for bit in range(C_BITS)) or (0,), C_BITS)
tensor_program = ProjectedProgram(projection, (), (), {"roles": 1, "width": 1 << C_BITS, "role_bits": 0})
rng = np.random.default_rng(SEED)
values = jnp.asarray(rng.normal(size=1 << C_BITS) + 1j * rng.normal(size=1 << C_BITS), dtype=DTYPE)
result, timings = timed(compile_c_tensor(C_BITS), values)
reference = compile_projected_target(tensor_program)(values[None, :])[0].block_until_ready()
print({"operator": f"C tensor {C_BITS}", **errors(result, reference), **timings})


In [ ]:
if not RUN_FULL_PROJECTED:
    print("Full projected replay disabled. Set RUN_FULL_PROJECTED=True to run it.")
else:
    if PROJECTED_BITS not in (1, 4):
        raise ValueError("Use PROJECTED_BITS=1 or 4 for this bounded full-network experiment")
    p = build_projected_program(h=4, bits=PROJECTED_BITS, seed=SEED)
    roles, width = p.metadata["roles"], p.metadata["width"]
    # Dirty dyadic Gaussian values are exactly representable in both input precisions.
    exact_values = tuple(tuple(QI(Fraction((17*r + 5*a) % 29 - 14, 8),
                                 Fraction((11*r + 7*a) % 31 - 15, 16))
                              for a in range(width)) for r in range(roles))
    def complex_rows(rows):
        return [[complex(float(z.real), float(z.imag)) for z in row] for row in rows]
    values = jnp.asarray(complex_rows(exact_values), dtype=DTYPE)
    reference = np.asarray(complex_rows(projected_target_exact(p, exact_values)))
    print({key: p.metadata[key] for key in ("h", "roles", "width", "record_count", "status")})
    for label, function in (("independent_target", compile_projected_target(p)),
                            ("direct_shear_replay", compile_projected(p, False)),
                            ("three_C_shear_replay", compile_projected(p, True))):
        result, timings = timed(function, values)
        print({"method": label, **errors(result, reference), **timings})


## 4. Actual positive-sign DFT baselines

All references here use $y_k=\sum_j x_j\exp(+2\pi i j k/n)$, unnormalized.
`conj(fft(conj(x)))` and `n*ifft(x)` match this convention; `ifft(x)` alone has a $1/n$ scale, and `fft(x)` uses the opposite sign. Radix-two and Bluestein are classical numerical baselines, not the paper's saving algorithm. First-call time includes compilation; warm samples are fenced and exclude transfers.


In [ ]:
if type(DFT_LENGTH) is not int or not 1 <= DFT_LENGTH <= 1024:
    raise ValueError("Use an integer DFT_LENGTH between 1 and 1024 for dense calibration")
if type(WARM_REPEATS) is not int or WARM_REPEATS < 1:
    raise ValueError("WARM_REPEATS must be positive")

def positive_dft_reference(x):
    x = np.asarray(x, dtype=np.complex128)
    j = np.arange(len(x))
    return np.exp(2j * np.pi * j[:, None] * j[None, :] / len(x)) @ x

n = DFT_LENGTH
rng = np.random.default_rng(SEED)
x = jnp.asarray(rng.normal(size=n) + 1j * rng.normal(size=n), dtype=DTYPE)
reference = positive_dft_reference(x)
methods = {"jax_conjugate_forward": jax.jit(lambda x: jnp.conj(jnp.fft.fft(jnp.conj(x)))),
           "jax_ifft_times_n": jax.jit(lambda x: n * jnp.fft.ifft(x)),
           "Bluestein": compile_bluestein(n)}
if n & (n - 1) == 0:
    methods["radix_two"] = compile_radix2(n)
for label, function in methods.items():
    result, timings = timed(function, x)
    print({"n": n, "method": label, **errors(result, reference), **timings})
print({"normalization_control_ifft_only": errors(jnp.fft.ifft(x), reference)})
print({"sign_control_fft": errors(jnp.fft.fft(x), reference)})


## 5. Companion Newton factorization: a local dense identity

For the positive Fourier matrix, the paper's factors satisfy
$F=N\,\mathrm{diag}(N_{00},\ldots,N_{n-1,n-1})^{-1}N^T$.
The transpose is **ordinary**, without conjugation. This experiment computes those factors directly in the selected floating-point precision; it performs dense $O(n^2)$ work and does not implement the shallow Toeplitz compiler, shared RAM preparation or global batching scheduler.

The factor identity has exact Lean component proofs. Numerical agreement here is a separate test; cancellation and large factor entries can make it unstable as $n$ grows. The all-length uniform theorem remains unfinished. Keep `NEWTON_LENGTH <= 64`; the module enforces this default dense-size guard.


In [ ]:
try:
    from exact_fourier.jax_newton import compile_newton, newton_factors
except ImportError:
    print("jax_newton is not present in this checkout yet; the other experiments remain runnable.")
else:
    n = NEWTON_LENGTH
    if type(n) is not int or not 1 <= n <= 64:
        raise ValueError("Use 1 <= NEWTON_LENGTH <= 64")
    rng = np.random.default_rng(SEED)
    x = jnp.asarray(rng.normal(size=n) + 1j * rng.normal(size=n), dtype=DTYPE)
    reference = positive_dft_reference(x)
    N, diagonal = newton_factors(n, dtype=COMPLEX_DTYPE)
    reconstructed = np.asarray(N) @ np.diag(1 / np.asarray(diagonal)) @ np.asarray(N).T
    fourier = np.exp(2j * np.pi * np.arange(n)[:, None] * np.arange(n)[None, :] / n)
    result, timings = timed(compile_newton(n), x)
    print({"n": n, "Newton_factor_matrix_error": errors(reconstructed, fourier),
           "maximum_factor_entry": float(np.max(np.abs(np.asarray(N)))),
           "Newton_action_error": errors(result, reference), **timings})
    # Replacing N.T by its conjugate transpose is a competing, incorrect formula.
    wrong = np.asarray(N) @ np.diag(1 / np.asarray(diagonal)) @ np.asarray(N).conj().T
    print({"conjugate_transpose_control": errors(wrong, fourier)})
    baseline, timings = timed(compile_bluestein(n), x)
    print({"method": "Bluestein_same_DFT", **errors(baseline, reference), **timings})


## Recorded evidence and boundaries

The cell below reads existing receipts; it does not rerun Lean or claim a new CUDA run. A failed accuracy receipt is useful numerical evidence and must remain visible. Source hashes identify the snapshot tested. Compare fresh notebook outcomes against that snapshot before attributing differences to hardware.

- [Lean construction audit](../docs/constructive-seed-audit.md): the explicit saving word and count are kernel-checked. The Python lazy generator's equivalence to that exact Lean word has not been proved.
- [Uniform proof contract](../docs/uniform-proof-contract.md): the companion's complete all-length fixed-program algorithm is still unproved; component proofs do not establish the final theorem.
- [JAX replication receipts](../verification/jax-replication/): actual CPU/CUDA numerical observations, not evidence that exact-complex Lean uses machine precision.

There is no practical fast-DFT or speedup claim for these paper constructions. Classical FFT baselines compute the same DFT in section 4; the C tensor and projected quotient in section 3 are different operators.


In [ ]:
receipt_dir = REPO / "verification/jax-replication"
for name in ("fft.json", "projection.json", "shear.json", "normalization.json", "newton.json"):
    path = receipt_dir / name
    if path.is_file():
        report = json.loads(path.read_text())
        print({"receipt": name, "passed": report.get("passed", report.get("status")),
               "host": report.get("host"), "devices": report.get("devices"),
               "started_utc": report.get("started_utc")})
manifest = receipt_dir / "source-manifest.json"
if manifest.is_file():
    recorded = json.loads(manifest.read_text())
    for relative in ("src/exact_fourier/jax_backend.py", "src/exact_fourier/jax_fft.py", "src/exact_fourier/jax_newton.py"):
        path = REPO / relative
        if path.is_file():
            actual = hashlib.sha256(path.read_bytes()).hexdigest()
            print({"source": relative, "sha256": actual,
                   "matches_recorded_snapshot": actual == recorded[relative] if relative in recorded else "not recorded"})
